# MLP PyTorch — Credit Risk

Neural network baseline using PyTorch. Same data split as notebook 03 for fair comparison.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import mlflow

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
mlflow.set_tracking_uri(f"sqlite:///{project_root}/mlflow.db")

df = pd.read_csv('../data/processed/credit_risk_clean.csv')
print(f'Shape: {df.shape}')

Shape: (149986, 12)


## 1. Prepare Data

In [2]:
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Neural nets need scaled features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Convert to tensors
X_train_t = torch.FloatTensor(X_train_s)
y_train_t = torch.FloatTensor(y_train.values)
X_test_t = torch.FloatTensor(X_test_s)
y_test_t = torch.FloatTensor(y_test.values)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)

print(f'Train: {len(X_train_t)} | Test: {len(X_test_t)}')
print(f'Features: {X_train_t.shape[1]}')

Train: 119988 | Test: 29998
Features: 11


## 2. Define Model

In [3]:
class CreditMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

model = CreditMLP(X_train_t.shape[1])
print(model)

CreditMLP(
  (net): Sequential(
    (0): Linear(in_features=11, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


## 3. Train

In [4]:
# Handle class imbalance via pos_weight
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(X_batch)

    epoch_loss /= len(train_ds)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:02d}/{EPOCHS} — loss: {epoch_loss:.4f}')

Epoch 05/20 — loss: 0.8974
Epoch 10/20 — loss: 0.8923
Epoch 15/20 — loss: 0.8892
Epoch 20/20 — loss: 0.8856


## 4. Evaluate

In [5]:
model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    y_prob_mlp = torch.sigmoid(logits).numpy()
    y_pred_mlp = (y_prob_mlp >= 0.5).astype(int)

auc_mlp = roc_auc_score(y_test, y_prob_mlp)
print(f'AUC: {auc_mlp:.4f}')
print()
print(classification_report(y_test, y_pred_mlp, target_names=['Paid', 'Defaulted']))

AUC: 0.8675

              precision    recall  f1-score   support

        Paid       0.98      0.76      0.86     27993
   Defaulted       0.20      0.82      0.32      2005

    accuracy                           0.76     29998
   macro avg       0.59      0.79      0.59     29998
weighted avg       0.93      0.76      0.82     29998



## 5. Log to MLflow

In [6]:
mlflow.set_experiment('credit-risk')

with mlflow.start_run(run_name='mlp-pytorch'):
    mlflow.log_param('model_type', 'MLP')
    mlflow.log_param('hidden_layers', '64-32')
    mlflow.log_param('dropout', '0.3-0.2')
    mlflow.log_param('epochs', EPOCHS)
    mlflow.log_param('lr', 1e-3)
    mlflow.log_param('batch_size', 512)
    mlflow.log_param('pos_weight', f'{pos_weight.item():.2f}')
    mlflow.log_metric('auc', auc_mlp)

    # Save model weights
    torch.save(model.state_dict(), '/tmp/mlp_credit.pt')
    mlflow.log_artifact('/tmp/mlp_credit.pt')

    print(f'Logged. AUC: {auc_mlp:.4f}')

Logged. AUC: 0.8675
